# PS 2 — Iris Flower (UCI / sklearn)

**DWM practical | Run cells top-to-bottom.**

All datasets are read from `../data/` and results saved in `../outputs/`. Run `python download_datasets.py` in the repo root beforehand.

**Goal:** examine Iris, remove duplicates, encode species, then compare Min-Max and Z-score transforms. The official Iris dataset is loaded from scikit-learn (offline).

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler, StandardScaler
DATA = Path('../data')
OUT = Path('../outputs'); OUT.mkdir(exist_ok=True)

from sklearn.datasets import load_iris
iris = load_iris(as_frame=True)
df = iris.frame.copy()
df = df.rename(columns={'target':'species'})
df['species'] = df['species'].map(dict(enumerate(iris.target_names)))
before = df.copy()

In [ ]:
print('Shape:', df.shape)
print('Columns:', df.columns.tolist())
print('Dtypes:'); print(df.dtypes)
print('Missing values:'); print(df.isna().sum())
print('Duplicates:', df.duplicated().sum())
display(df.head())
display(df.describe(include='all').T)

In [ ]:
print('Data types:'); print(df.dtypes)
df = df.drop_duplicates().copy()
features = df.drop(columns=['species']).columns.tolist()
for c in features: df[c] = pd.to_numeric(df[c], errors='coerce').fillna(df[c].median())
df['species'] = df['species'].astype('string').str.lower().str.strip()
labels = {'setosa':0,'versicolor':1,'virginica':2}
df['species_encoded'] = df['species'].map(labels).astype(int)
print('Species mapping:',labels)

**Min-Max:** `(x−min)/(max−min)` gives values between 0 and 1. **Z-score:** `(x−mean)/std` gives approximately zero mean and unit variance. Keep each transform as separate columns so they can be compared.

In [ ]:
minmax = pd.DataFrame(MinMaxScaler().fit_transform(df[features]),columns=features,index=df.index)
zscore = pd.DataFrame(StandardScaler().fit_transform(df[features]),columns=features,index=df.index)
print('ORIGINAL:');display(df[features].head())
print('MIN-MAX:');display(minmax.head())
print('Z-SCORE:');display(zscore.head())
print('Z-score means:',zscore.mean().round(5).to_dict())
print('Z-score standard deviations:',zscore.std(ddof=0).round(5).to_dict())

In [ ]:
processed = minmax.add_suffix('_minmax').join(zscore.add_suffix('_zscore'))
processed['species_encoded'] = df['species_encoded']
print('Before:');display(before.head())
print('After:');display(processed.head())
processed.to_csv(OUT/'PS02_iris_processed.csv',index=False)